In [ ]:
import os
from pathlib import Path

os.environ["HF_TOKEN"] = "TOKEN"

import torch
from diffusers import Kandinsky6TI2VAPipeline
from diffusers.hooks import MagCacheConfig
from diffusers.models.transformers.transformer_kandinsky6 import (
    Kandinsky6FusedTransformerDecoderBlock,
    Kandinsky6TransformerDecoderBlock,
    Kandinsky6TransformerEncoderBlock,
)
from diffusers.utils import encode_video

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "dev" / "exports" / "diffusers").exists():
    REPO = REPO.parent
if not (REPO / "dev" / "exports" / "diffusers").exists():
    raise RuntimeError("Run this notebook from the K6 repository or one of its subdirectories.")

### Initialize pipeline

In [ ]:
# Keep generation and compile settings in one K6 YAML config. MagCache defaults are exported with the transformer.
GPU_ID = 2

MODELS = {
    "k6pro": "kandinskylab/Kandinsky-6.0-Pro-sft-5s-Diffusers",
    "k6pro_distill": "kandinskylab/Kandinsky-6.0-Pro-distill-5s-Diffusers",
}

model_name = "k6pro_distill"
model_path = MODELS[model_name]
expand_prompts = True
use_magcache = bool("distill" not in model_name)
use_compile = False
max_sequence_length = 1024
sample_fps = 24.0
seed = 1137

backend = "native" if use_compile else "_flash_3"

pipe = Kandinsky6TI2VAPipeline.from_pretrained(model_path, dtype=torch.bfloat16)
pipe.transformer.set_attention_backend(backend)

# Compile after loading the weights and before CPU offload. This applies the
# configured K6 block-level torch.compile strategy to the exported transformer.
if use_compile:
    torch.set_float32_matmul_precision('high')
    for block in pipe.transformer.modules():
        if isinstance(block, Kandinsky6FusedTransformerDecoderBlock):
            block.forward = torch.compile(block.forward)
        elif isinstance(block, (Kandinsky6TransformerEncoderBlock, Kandinsky6TransformerDecoderBlock)):
            block.forward = torch.compile(
                block.forward, mode="max-autotune-no-cudagraphs", dynamic=True
            )

is_piflow = pipe.scheduler.__class__.__name__ == "PiflowScheduler"
num_inference_steps = 16 if is_piflow else 50
guidance_scale = 1.0 if is_piflow else 5.0
print(f"Scheduler: {pipe.scheduler.__class__.__name__}; steps: {num_inference_steps}; guidance: {guidance_scale}")

### Run generation

In [ ]:
# NOTE: uses different mag_ratios
gen_mode = "t2va" # "t2va"

output_suffix = ""
output_suffix += "_piflow" if is_piflow else ""
output_suffix += "_expand" if expand_prompts else ""
output_suffix += "_compile" if use_compile else ""
output_suffix += "_magcache" if use_magcache else ""
output_path = REPO / "outputs" / "diffusers" / model_name
output_path.mkdir(parents=True, exist_ok=True)

# Use Diffusers' official MagCacheConfig and model enable_cache API. K6
# stores conditional/unconditional coefficients interleaved, so the
# configuration counts both CFG forwards.
magcache = pipe.transformer.config.get("magcache")
if use_magcache and magcache is not None:
    mag_ratios = magcache["mag_ratios"]
    if isinstance(mag_ratios, dict):
        mag_ratios = mag_ratios[gen_mode]
    diffusers_magcache = MagCacheConfig(
        threshold=magcache["threshold"],
        max_skip_steps=magcache["max_skip_steps"],
        retention_ratio=magcache["retention_ratio"],
        num_inference_steps=2 * num_inference_steps,
        mag_ratios=list(mag_ratios),
    )
    if pipe.transformer.is_cache_enabled:
        pipe.transformer.disable_cache()
    pipe.transformer.enable_cache(diffusers_magcache)
    print(f"Diffusers MagCache enabled: {pipe.transformer.is_cache_enabled}")
    print(f"MagCache coefficients ({gen_mode}): {len(mag_ratios)}; threshold: {magcache['threshold']}")
elif getattr(pipe.transformer, "is_cache_enabled", False):
    pipe.transformer.disable_cache()

# Model CPU offload keeps one active component on GPU at a time.
pipe.enable_model_cpu_offload(gpu_id=GPU_ID)

print(f"Qwen context length: {max_sequence_length}")
print(f"Audio sample rate: {pipe.audio_sample_rate}; video FPS: {sample_fps}")

In [ ]:
batch_size = 2
prompt = (
    "A news anchor delivers the evening news in a clear, standard, and neutral tone, saying: "
    "<S>Good evening, ladies and gentlemen, and welcome to the evening news.<E>"
)
negative_prompt = "blurry, distorted, low quality, noisy audio"

prompts = [prompt] * batch_size
negative_prompts = [negative_prompt] * batch_size
images = None

# NOTE: use example image for I2VA mode
if gen_mode == "i2va":
    image = REPO / "assets" / "i2va_input.png"
    print(f"reference image: {image}")
    images = [str(image)] * batch_size


result = pipe(
    prompt=prompts,
    image=images,
    negative_prompt=negative_prompts,
    height=480,
    width=864,
    num_frames=121,
    sample_fps=sample_fps,
    num_inference_steps=num_inference_steps,
    max_sequence_length=max_sequence_length,
    guidance_scale=guidance_scale,
    sample_audio=True,
    expand_prompts=True,
    generator=torch.Generator().manual_seed(seed),
    output_type="pt",
)
frames = result.frames
audios = result.audio
print(f"decoded frames: {tuple(frames.shape)}")
print(f"audio streams: {len(audios)}, samples in first stream: {audios[0].shape[0]}")

if use_magcache and magcache is not None:
    print("MagCache run complete")

In [ ]:
final_output_dir = output_path / f"bs{batch_size}"
final_output_dir.mkdir(parents=True, exist_ok=True)

for i, (video, audio) in enumerate(zip(frames, audios)):
    batch_suffix = f"_batchid{i}" if batch_size > 1 else ""
    video_path = final_output_dir / f"kandinsky6_{gen_mode}_{num_inference_steps}steps{output_suffix}{batch_suffix}.mp4"
    video = video.permute(1, 2, 3, 0)
    audio_samples = torch.as_tensor(audio)[:, None].repeat(1, 2)
    encode_video(
        video,
        fps=int(sample_fps),
        output_path=str(video_path),
        audio=audio_samples,
        audio_sample_rate=int(pipe.audio_sample_rate),
    )
    print(f"saved: {video_path}")